# Cellular Inference Mesh — Reproducible Replication Notebook

**Software DOI (concept):** [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.20108648.svg)](https://doi.org/10.5281/zenodo.20108648) &nbsp; **Repository:** <https://github.com/ulissesflores/cellular-inference-mesh> &nbsp; **License:** Apache-2.0

> **Saturated Pipeline Conjecture.** In edge-cloud LLM pipelines with $L_{\text{edge}} \approx L_{\text{cloud}}$, p99 latency is structurally insensitive to fallback-rate reductions — a finding that reframes optimization from routing to serial-path acceleration.

This notebook is a **third-party-reproducible audit trail** for the Salabim discrete-event simulation that backs the paper. Click **Runtime → Run all** and the notebook will:

1. Clone the repository and install pinned dependencies.
2. Run the 53-test pytest suite.
3. Execute a smoke simulation (~30 s) to confirm the runtime is healthy.
4. Verify SHA-256 bit-parity of source files and outputs against `output/experiment_provenance.json`.
5. Re-verify Theorem 1 (Pareto-efficiency) by computational enumeration.
6. Validate the Saturated Pipeline Conjecture numerically across four $L_{\text{edge}}/L_{\text{cloud}}$ regimes.
7. Print key empirical results (the same numbers cited in the paper's Table 2).
8. Render the five paper-ready figures inline.

For a full canonical run (300 replicates × 1800 s × 6 scenario-arms ≈ 31 min on 8 cores), uncomment the indicated cell below.


## 1. Environment setup

The notebook uses Python 3.14 (or any 3.12+) with the dependencies pinned in `requirements.txt`. Both Colab and local execution paths are supported below.


In [ ]:
# Clone the repository (skipped if already present locally).
import os
import subprocess
import sys

REPO_URL = "https://github.com/ulissesflores/cellular-inference-mesh.git"
WORKSPACE = "/content/cellular-inference-mesh" if os.path.exists("/content") else "."

if WORKSPACE != "." and not os.path.exists(WORKSPACE):
    subprocess.run(["git", "clone", REPO_URL, WORKSPACE], check=True)

os.chdir(WORKSPACE)
sys.path.insert(0, WORKSPACE)
print(f"Working directory: {os.getcwd()}")


In [ ]:
# Install pinned dependencies (editable install with dev extras for pytest).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# Confirm runtime versions match expected pins.
import matplotlib
import networkx
import numpy
import pandas
import salabim
import scipy

print(f"  salabim    {salabim.__version__}")
print(f"  numpy      {numpy.__version__}")
print(f"  scipy      {scipy.__version__}")
print(f"  matplotlib {matplotlib.__version__}")
print(f"  networkx   {networkx.__version__}")
print(f"  pandas     {pandas.__version__}")


## 2. Mathematical validation (53 pytest tests)

The pytest suite validates the canonical formulas of the integrated techniques before any simulation runs:

- Leviathan, Kalman & Matias (2023) — speculative-decoding speedup formula
- Little's Law — empirical validation across the simulation
- Pollaczek–Khinchine — M/G/1 saturation bound
- KV-cache reduction — multiplicative composition (38× target)
- Blast radius — combinatorial $\binom{8}{2} = 28$ exact

Expected output: `53 passed` in ~5.5 s.


In [ ]:
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-q", "--tb=short"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
assert "passed" in result.stdout, "pytest did not pass — replication aborted."


## 3. Canonical Salabim simulation

Pipeline matching paper §2.3. `seed_canonical = 42`, 3 scenarios × 300 replicates = 1800 records, 1800 s simulated per replicate.

Per-replicate parameters are sampled from a realistic operational range using `random.Random(42 + i)`:

- `lambda_voice_eps` ∈ [1.95, 2.05] events/s/AGV
- `n_agvs` = 10 (fixed, Jensen-safe)
- `partition_inject_pct` ∈ [1.5, 2.5] %
- `W_jitter_pct` ∈ [0.9, 1.1] (multiplicative on `uplink_rtt_std_ms`)

A reduced smoke run (~30 s) is executed below; the full canonical run (~31 min) is optional.


In [ ]:
# Smoke run (~30 s) — confirms the simulation engine is functional.

from src.config import ExperimentConfig
from src.simulation import run_replica

cfg = ExperimentConfig(sim_duration_s=60.0, scenario="nominal")
result = run_replica(cfg, proposed=True, replica_seed=42)
print(f"Smoke run complete: p99={result['p99_ms']:.1f} ms, "
      f"fallback={result['fallback_rate']*100:.1f}%")


In [ ]:
# Full canonical run (~31 min with 8 workers) — uncomment to reproduce paper hashes.
# !python -m src.main --replicates 300 --duration 1800 --workers 8

print("Full canonical run is required to regenerate output/results.json with the hashes")
print("expected by the verification cell below. Uncomment the line above to execute.")


## 4. Bit-parity verification (SHA-256 chain)

Reproducibility here is mathematical, not rhetorical: given `seed_canonical = 42` plus the pinned dependencies, re-execution produces bit-identical outputs.

The verification cell below loads the expected hashes **dynamically** from `output/experiment_provenance.json` (the source of truth) and compares them against the freshly computed SHA-256 of every source and output file. Any mismatch flips the corresponding line to ❌ — there is no place for hand-edited expected values.


In [ ]:
import hashlib
import json
from pathlib import Path


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

provenance = json.load(open("output/experiment_provenance.json"))

print("Source files (SHA-256):")
for fn, expected in provenance["source_files_sha256"].items():
    actual = sha256_file(f"src/{fn}")
    ok = "✅" if actual == expected else "❌"
    print(f"  {ok}  src/{fn:<20s}  {actual[:16]}...")

print()
print("Outputs (SHA-256):")
for label, key, path in [
    ("results.json       ", "results_sha256", "output/results.json"),
    ("raw_replicas.jsonl ", "raw_replicas_sha256", "output/raw_replicas.jsonl"),
]:
    actual = sha256_file(path)
    expected = provenance[key]
    ok = "✅" if actual == expected else "❌"
    print(f"  {ok}  {label}  {actual[:16]}...")

print()
print(f"Package version:  {provenance['version']}")
print(f"Git commit:       {provenance['git_commit_sha'][:12]}")
print(f"Generated (UTC):  {provenance['timestamp_utc']}")


## 5. Theorem 1 — computational verification

The script enumerates the $2^9 = 512$ sub-compositions of the 9-technique design space, identifies the 256 valid sub-compositions that satisfy the mandatory payload-gating constraint, computes multiplicative scores across the 5 axes, and confirms that **zero sub-compositions Pareto-dominate** the complete CIM-PCE composition.

A Monte-Carlo sensitivity analysis (1000 realizations, ±20% perturbation on factor matrix) confirms 100% robustness.

The verification is extended to 7 external concrete architectures: CIM-PCE Pareto-dominates 6 of them and remains Pareto-incomparable with EAGLE-3 in single-request analytical latency only.


In [ ]:
result = subprocess.run(
    ["python", "scripts/pareto_proof.py"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])


## 6. Saturated Pipeline Conjecture — numerical validation

Under edge saturation ($\rho_{\text{edge}} \to 1$, $L_{\text{edge}} \approx L_{\text{cloud}}$), $\mathrm{p99}(L_{\text{total}})$ is structurally insensitive to $p_{fb}$ via the mixture decomposition

$$
\mathrm{p99}(L_{\text{total}}) \approx p_{fb} \cdot \mathrm{p99}(L_{\text{cloud}}) + (1 - p_{fb}) \cdot \mathrm{p99}(L_{\text{edge}})
$$

The script below validates four regimes $L_{\text{edge}} / L_{\text{cloud}} \in \{0.1, 0.5, 0.91, 1.0\}$, confirming that the empirical observation falls precisely on the $0.91$ curve.


In [ ]:
result = subprocess.run(
    ["python", "scripts/saturated_pipeline_plot.py"],
    capture_output=True, text=True,
)
print(result.stdout[-1500:])


## 7. Key empirical results (paper Table 2)

The values printed below are the same numbers cited in the paper's results table. Any mismatch with the paper indicates the canonical simulation has not been re-executed (the smoke run alone does not regenerate `output/results.json`).


In [ ]:
results = json.load(open("output/results.json"))

print("Nominal scenario (n=300 replicates with parametric variation; 95% CI):")
for arm in ["baseline", "proposed"]:
    r = results[f"{arm}__nominal"]
    print(f"  {arm}:")
    print(f"    p99       = {r['p99_ms']['mean']:.2f} ± {r['p99_ms']['ci95']:.2f} ms")
    print(f"    fallback  = {r['fallback_rate']['mean']*100:.2f}% ± {r['fallback_rate']['ci95']*100:.3f}%")
    print(f"    cost      = US$ {r['cost_usd']['mean']:.5f} ± US$ {r['cost_usd']['ci95']:.6f}")

f_p99      = results['baseline__nominal']['p99_ms']['mean']    / results['proposed__nominal']['p99_ms']['mean']
f_fallback = results['baseline__nominal']['fallback_rate']['mean'] / results['proposed__nominal']['fallback_rate']['mean']
f_cost     = results['baseline__nominal']['cost_usd']['mean']  / results['proposed__nominal']['cost_usd']['mean']

print()
print("Empirical reduction factors (paper Table 2):")
print(f"  p99       reduction: {f_p99:.2f}× (paper reports 1.10×)")
print(f"  fallback  reduction: {f_fallback:.2f}× (paper reports 7.32×)")
print(f"  cost      reduction: {f_cost:.2f}× (paper reports 19.76×)")


## 8. Visualization of the five paper-ready figures

The figures rendered below are the same ones embedded in the paper. They are produced from the canonical simulation outputs via `python -m src.report` and `python scripts/figs_math_diagrams.py`.


In [ ]:
from IPython.display import Image, Markdown, display

essential_figs = [
    ("fig8_speculative_speedup.png",         "Figure 1 — Speculative decoding speedup curves (Leviathan, Kalman & Matias, 2023)."),
    ("fig7_roofline.png",                     "Figure 2 — Roofline analysis for Llama 3 8B INT4 on Intel Xeon with DDR5-5600."),
    ("fig12_decision_tree_payload_gating.png","Figure 3 — Payload-gating decision tree under Zero Trust."),
    ("fig2_ecdf_latency.png",                 "Figure 4 — ECDF of p99 latency across 300 Monte-Carlo replicates (two-panel layout)."),
    ("fig6_pacelc_box.png",                   "Figure 5 — PACELC validation box plot (nominal + partition + burst)."),
]

for fname, caption in essential_figs:
    display(Markdown(f"**{caption}**"))
    display(Image(filename=f"output/{fname}"))


## Conclusion of the audit

If every checkpoint above passes (53/53 pytest, SHA-256 hashes matching `output/experiment_provenance.json`, Theorem 1 verified, Conjecture validated, results matching the paper's Table 2, figures rendering), then the local execution faithfully reproduces the canonical results.

To cite this software, please use the BibTeX block in the repository's `README.md` or the machine-readable `CITATION.cff`:

```bibtex
@software{flores_cellular_inference_mesh_2026,
  author       = {Flores, Carlos Ulisses},
  title        = {{Cellular Inference Mesh: Salabim DES of Edge-Cloud LLM Inference under PACELC}},
  year         = 2026,
  publisher    = {Zenodo},
  version      = {0.3.0},
  doi          = {10.5281/zenodo.20108648},
  url          = {https://doi.org/10.5281/zenodo.20108648},
}
```

For scientific contact, please use [ORCID 0000-0002-6034-7765](https://orcid.org/0000-0002-6034-7765).
